# 10 - Within-Session Temporal Consistency and Robustness Analysis (WORKING COPY - TARGETED FIX)
**Master Workspace:** `MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/08_COLAB_NOTEBOOKS`  
**Original Notebook Preserved:** `10_Within_Session_Temporal_Robustness.ipynb`  
**Analysis Title:** **Within-Session Temporal Consistency and Robustness Analysis**  

### Targeted Fixes & Locked Temporal Partition Applied in This Notebook:
1. **Locked Temporal Partition (`3 x 300s Primary Segments + 38.84s Residual Recording`):**
   - `Segmen_1 (00:00-05:00)`: frames `0..7,499` (`7,500` frames = `300.00 s` at `25 FPS`, theoretical ceiling `6,300.00 s`).
   - `Segmen_2 (05:00-10:00)`: frames `7,500..14,999` (`7,500` frames = `300.00 s` at `25 FPS`, theoretical ceiling `6,300.00 s`).
   - `Segmen_3 (10:00-15:00)`: frames `15,000..22,499` (`7,500` frames = `300.00 s` at `25 FPS`, theoretical ceiling `6,300.00 s`).
   - `Residual_Recording (15:00-15:38.84)`: frames `22,500..23,470` (`971` frames = `38.84 s` at `25 FPS`, theoretical ceiling `815.64 s`) - recorded separately and **not** merged into `Segmen_3`.
   - `Full_Session (00:00-15:38.84)`: frames `0..23,470` (`23,471` frames = `938.84 s` at `25 FPS`, theoretical ceiling `19,715.64 s`).
2. **Fixed Schema Column Name (`BUG-02`):** `load_and_deduplicate_identity_assignments()` reads `assignment_status` (matching `src/identity.py` lines `392` and `844`, with fallback to `mapping_status`) and performs deterministic deduplication on `(frame, student_id)`.
3. **Direct `21 Students x 4 Activities` Matrix Construction & IPF Comparison (`BUG-09`):** `build_direct_frame_counted_matrices_and_profiles()` builds the `21 x 4` activity duration matrices at `25 FPS` directly from `identity_assignments.csv`, and `compare_direct_vs_ipf_profiles()` computes both **absolute differences (seconds)** and **relative differences (%)** against Biproportional IPF.
4. **Self-Contained Per-Segment `AHP-SAW` Ranking, Spearman $\rho$, Kendall $\tau$, and Holm-Bonferroni Correction (`BUG-09`):** Implemented directly in `Cell 5` & `Cell 7` via `compute_ahp_saw_segment_rankings()` and `compute_temporal_concordance_holm_bonferroni()`, locked behind `ENABLE_TEMPORAL_EXECUTION = False`.


In [1]:
# ==============================================================================
# CENTRALIZED GOOGLE DRIVE PATHS, EXECUTION LOCKS & MULTI-DETECTOR SELECTOR
# ==============================================================================
import json
import logging
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Select Active Detector Model for Within-Session Temporal Robustness (3x300s + 38.84s Residual):
#   "yolo11m"       -> Primary YOLO11m baseline
#   "yolov8m"       -> YOLOv8m baseline (reads/builds from multi_detector_pipeline/yolov8m/)
#   "rfdetr_small"  -> RF-DETR-Small ("S", reads/builds from multi_detector_pipeline/rfdetr_small/)
#   "rfdetr_large"  -> RF-DETR-Large ("L", reads/builds from multi_detector_pipeline/rfdetr_large/)
#   "ALL_AVAILABLE" -> Automatically builds missing matrices from any existing detections.csv and runs all ready detectors!
ACTIVE_DETECTOR_MODEL = "ALL_AVAILABLE"

# Set AUTO_RUN_MISSING_VIDEO_INFERENCE = True if you want Notebook 10 to also run 23,471-frame
# video inference automatically when detections.csv has not been generated in Notebook 04 yet
AUTO_RUN_MISSING_VIDEO_INFERENCE = False

ENABLE_TEMPORAL_EXECUTION = True

ENABLE_ROBOFLOW_STAGING_DOWNLOAD = False
ENABLE_DATASET_REPLACEMENT = False
ENABLE_TRAINING = False
ENABLE_MODEL_EVALUATION = False
ENABLE_FULL_VIDEO_INFERENCE = False
ENABLE_TRACKER_REPLAY = False
ENABLE_MCDM_EVALUATION = False
ENABLE_IDENTITY_ABLATION = False
ENABLE_SEAT_PERTURBATION_SIMULATION = False
ENABLE_MONTE_CARLO = False
ENABLE_GPU_EXPERIMENTS = False

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger("YOLO11_IJITCS_SyncedColab")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MASTER_ROOT = Path(
    "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026"
)
DATASET_ROOT = MASTER_ROOT / "03_DATASET_AUDIT" / "YOLO11_DATASET_UNZIPPED"
DATA_YAML_PATH = DATASET_ROOT / "data_colab.yaml"
REPO_ROOT = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "YOLO11_Repository"
MODULES_DIR = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "Reconstructed_Research_Modules"
VIDEO_PATH = (
    MASTER_ROOT
    / "02_ORIGINAL_RESEARCH_ARCHIVE"
    / "Original_Videos"
    / "video_part001.mp4"
)
CHECKPOINT_DIR = MASTER_ROOT / "04_MODEL_AND_CONFIGURATION" / "Checkpoints"
OUTPUT_DIR = MASTER_ROOT / "09_REVALIDATED_EXPERIMENT_OUTPUTS"
OUTPUT_PATH = OUTPUT_DIR
MULTI_DETECTOR_PIPELINE_DIR = OUTPUT_DIR / "multi_detector_pipeline"

for _p in (REPO_ROOT, MODULES_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

from src.paths_config import get_detector_pipeline_paths
from src.detector_adapters_v2_aligned import ensure_detector_downstream_csvs_ready

def _resolve_existing(candidates: list[Path]) -> Path:
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

TRACKING_VAL_DIR = MASTER_ROOT / "06_TRACKING_AND_IDENTITY_VALIDATION"
STUDENT_SEATS_PATH = _resolve_existing([
    REPO_ROOT / "data" / "student_seats.csv",
    TRACKING_VAL_DIR / "student_seats.csv",
])

def get_active_detector_targets(selector: str, auto_build: bool = True) -> list[dict[str, Path]]:
    norm = str(selector).strip().upper()
    candidate_keys = (
        ["yolo11m", "yolov8m", "rfdetr_small", "rfdetr_large"]
        if norm == "ALL_AVAILABLE"
        else [selector]
    )
    targets = []
    for k in candidate_keys:
        if auto_build and MASTER_ROOT.exists():
            prep = ensure_detector_downstream_csvs_ready(
                detector_key=k,
                master_root=MASTER_ROOT,
                seats_csv_path=STUDENT_SEATS_PATH,
                video_path=VIDEO_PATH,
                fps=25.0,
                total_video_frames=23471,
                auto_run_video_inference=AUTO_RUN_MISSING_VIDEO_INFERENCE,
            )
            p = prep["paths"]
            if prep["ready"]:
                targets.append(p)
            else:
                logger.info(
                    "Skipping detector '%s' in Notebook 10 (%s). Run Notebook 04 for '%s' or set AUTO_RUN_MISSING_VIDEO_INFERENCE = True.",
                    p["detector_key"],
                    prep["status"],
                    p["detector_key"],
                )
        else:
            p = get_detector_pipeline_paths(k, master_root=MASTER_ROOT)
            if p["duration_matrix_csv"].exists() or p["detections_csv"].exists():
                targets.append(p)
    return targets or [get_detector_pipeline_paths("yolo11m", master_root=MASTER_ROOT)]

_preview_targets = get_active_detector_targets(ACTIVE_DETECTOR_MODEL, auto_build=False)
DETECTIONS_CSV_PATH = _preview_targets[0]["detections_csv"]
TRACK_SUMMARY_CSV_PATH = _preview_targets[0]["track_summary_csv"]
TRACK_TO_STUDENT_CSV_PATH = _preview_targets[0]["track_to_student_csv"]
STUDENT_AUDIT_CSV_PATH = _preview_targets[0]["student_audit_csv"]
DURATION_MATRIX_CSV_PATH = _preview_targets[0]["duration_matrix_csv"]
IDENTITY_ASSIGNMENTS_CSV_PATH = _preview_targets[0]["identity_assignments_csv"]

logger.info("Active Detector Selector in Notebook 10: %s | Candidates found: %s", ACTIVE_DETECTOR_MODEL, [t["detector_key"] for t in _preview_targets])

Mounted at /content/drive


## Step 2: Roboflow Stratified Random Split Verification (Condition 3)


In [2]:
ROBOFLOW_SPLIT_VERIFICATION = [
    {"Prefix_Series": "Campuran<N>_png", "Train": 515, "Valid": 133, "Test": 77, "Unique_Frames": 725, "Index_Span": "1..732", "Valid_Delta1_In_Train_Pct": 89.47, "Test_Delta1_In_Train_Pct": 94.81, "Split_Verdict": "Condition 3: Stratified Random Interleaved"},
    {"Prefix_Series": "Angkat_tangan<N>_png", "Train": 403, "Valid": 100, "Test": 61, "Unique_Frames": 541, "Index_Span": "1..560", "Valid_Delta1_In_Train_Pct": 80.00, "Test_Delta1_In_Train_Pct": 78.69, "Split_Verdict": "Condition 3: Stratified Random Interleaved"},
    {"Prefix_Series": "Tidur<N>_png", "Train": 312, "Valid": 118, "Test": 37, "Unique_Frames": 467, "Index_Span": "1..478", "Valid_Delta1_In_Train_Pct": 84.75, "Test_Delta1_In_Train_Pct": 83.78, "Split_Verdict": "Condition 3: Stratified Random Interleaved"},
    {"Prefix_Series": "Berdiri<N>_png", "Train": 143, "Valid": 42, "Test": 21, "Unique_Frames": 206, "Index_Span": "1..680", "Valid_Delta1_In_Train_Pct": 88.10, "Test_Delta1_In_Train_Pct": 95.24, "Split_Verdict": "Condition 3: Stratified Random Interleaved"},
    {"Prefix_Series": "TOTAL / OVERALL", "Train": 1373, "Valid": 393, "Test": 196, "Unique_Frames": 1939, "Index_Span": "1..732", "Valid_Delta1_In_Train_Pct": 85.50, "Test_Delta1_In_Train_Pct": 87.76, "Split_Verdict": "NOT Chronological Holdout (0% exact overlap)"},
]
split_audit_df = pd.DataFrame(ROBOFLOW_SPLIT_VERIFICATION)
if ENABLE_TEMPORAL_EXECUTION:
    split_audit_df.to_csv(OUTPUT_PATH / "roboflow_stratified_split_verification.csv", index=False)
    print(split_audit_df.to_string(index=False))
else:
    logger.info("[LOCKED] Roboflow split audit table ready (set ENABLE_TEMPORAL_EXECUTION = True in Cell 1 to export).")


       Prefix_Series  Train  Valid  Test  Unique_Frames Index_Span  Valid_Delta1_In_Train_Pct  Test_Delta1_In_Train_Pct                                Split_Verdict
     Campuran<N>_png    515    133    77            725     1..732                      89.47                     94.81   Condition 3: Stratified Random Interleaved
Angkat_tangan<N>_png    403    100    61            541     1..560                      80.00                     78.69   Condition 3: Stratified Random Interleaved
        Tidur<N>_png    312    118    37            467     1..478                      84.75                     83.78   Condition 3: Stratified Random Interleaved
      Berdiri<N>_png    143     42    21            206     1..680                      88.10                     95.24   Condition 3: Stratified Random Interleaved
     TOTAL / OVERALL   1373    393   196           1939     1..732                      85.50                     87.76 NOT Chronological Holdout (0% exact overlap)


## Step 3: Within-Session Temporal Consistency and Robustness Engine (`3 x 300s Segments + 38.84s Residual + Full Session`)


In [3]:
# ==============================================================================
# WITHIN-SESSION TEMPORAL CONSISTENCY & ROBUSTNESS ENGINE (IMPORTED FROM SRC/)
# ==============================================================================
from src.temporal_analysis import compare_representations, per_entity_rank_table
from src.temporal_consistency_aligned import (
    CRITERIA_COLS,
    FPS_LOCKED,
    LOCKED_TEMPORAL_WINDOWS,
    TOTAL_FRAMES_LOCKED,
    build_direct_frame_counted_matrices_and_profiles,
    compare_direct_vs_ipf_profiles,
    compute_ahp_saw_segment_rankings,
    compute_biproportional_ipf_3x300_plus_residual,
    compute_temporal_concordance_holm_bonferroni,
    holm_bonferroni_step_down,
    load_and_deduplicate_identity_assignments,
)

windows_preview_df = pd.DataFrame(LOCKED_TEMPORAL_WINDOWS)
logger.info("Loaded %d locked within-session temporal windows from src.temporal_consistency_aligned.", len(windows_preview_df))


## Step 4: Guarded Execution of Direct Frame-Counted Aggregation vs Biproportional IPF, Per-Segment AHP-SAW Rankings & Holm-Bonferroni Concordance (`ENABLE_TEMPORAL_EXECUTION = False`)


In [4]:
# ==============================================================================
# GUARDED EXECUTION CONTROLLER FOR NOTEBOOK 10 (MULTI-DETECTOR SUPPORT)
# ==============================================================================
if ENABLE_TEMPORAL_EXECUTION:
    _all_temporal_concordance = []
    for _t in get_active_detector_targets(ACTIVE_DETECTOR_MODEL):
        _dkey = _t["detector_key"]
        _dur_path = _t["duration_matrix_csv"]
        _t2s_path = _t["track_to_student_csv"]
        _ts_path = _t["track_summary_csv"]
        _id_path = _t["identity_assignments_csv"]
        _out_dir = OUTPUT_DIR if _dkey == "yolo11m" else (_t["pipeline_dir"] / "temporal_robustness")
        _out_dir.mkdir(parents=True, exist_ok=True)

        if not _dur_path.exists() or not _t2s_path.exists() or not _ts_path.exists():
            raise FileNotFoundError(
                f"Missing supporting CSVs for detector '{_dkey}' at {_t['pipeline_dir']}. "
                f"Run Notebook 04 with ACTIVE_DETECTOR_MODEL = '{_dkey}' and ENABLE_FULL_VIDEO_INFERENCE = True first."
            )

        dur_full_df = pd.read_csv(_dur_path)
        t2s_df = pd.read_csv(_t2s_path)
        ts_df = pd.read_csv(_ts_path)

        # 1. Compute Biproportional IPF under the locked 3x300s + 38.84s Residual partition
        ipf_profiles_df, ipf_matrices = compute_biproportional_ipf_3x300_plus_residual(
            dur_full_df=dur_full_df,
            track_to_student_df=t2s_df,
            track_summary_df=ts_df,
            fps=FPS_LOCKED,
        )
        ipf_profiles_df.insert(0, "detector_model", _dkey)
        if _dkey == "yolo11m":
            ipf_profiles_df.drop(columns=["detector_model"]).to_csv(_out_dir / "corrected_within_session_segment_profiles.csv", index=False)
        ipf_profiles_df.to_csv(_out_dir / f"corrected_within_session_segment_profiles_{_dkey}.csv", index=False)

        # 2. Compute direct frame-counted 21x4 matrices from identity_assignments.csv and compare with IPF
        if _id_path.exists():
            df_primary = load_and_deduplicate_identity_assignments(_id_path)
            direct_profiles_df, direct_matrices = build_direct_frame_counted_matrices_and_profiles(
                df_primary=df_primary,
                student_ids=sorted(dur_full_df["entity"].astype(str).unique().tolist()),
                fps=FPS_LOCKED,
            )
            direct_profiles_df.insert(0, "detector_model", _dkey)
            if _dkey == "yolo11m":
                direct_profiles_df.drop(columns=["detector_model"]).to_csv(_out_dir / "direct_frame_counted_segment_profiles.csv", index=False)
            direct_profiles_df.to_csv(_out_dir / f"direct_frame_counted_segment_profiles_{_dkey}.csv", index=False)

            for wid, mat_w in direct_matrices.items():
                if _dkey == "yolo11m":
                    mat_w.reset_index().to_csv(_out_dir / f"direct_student_activity_matrix_{wid}.csv", index=False)
                mat_w.reset_index().to_csv(_out_dir / f"direct_student_activity_matrix_{wid}_{_dkey}.csv", index=False)

            diff_df = compare_direct_vs_ipf_profiles(
                direct_profiles_df.drop(columns=["detector_model"]),
                ipf_profiles_df.drop(columns=["detector_model"]),
            )
            diff_df.insert(0, "detector_model", _dkey)
            if _dkey == "yolo11m":
                diff_df.drop(columns=["detector_model"]).to_csv(_out_dir / "direct_vs_ipf_segment_comparison.csv", index=False)
            diff_df.to_csv(_out_dir / f"direct_vs_ipf_segment_comparison_{_dkey}.csv", index=False)
            active_matrices = direct_matrices
            print(f"\n=== DIRECT VS BIPROPORTIONAL IPF COMPARISON ({_dkey}) ===")
            print(diff_df.to_string(index=False))
        else:
            active_matrices = ipf_matrices

        # 3. Compute per-segment AHP-SAW rankings and Holm-Bonferroni concordance
        rankings_df = compute_ahp_saw_segment_rankings(active_matrices)
        concordance_df = compute_temporal_concordance_holm_bonferroni(rankings_df)
        rankings_df.insert(0, "detector_model", _dkey)
        concordance_df.insert(0, "detector_model", _dkey)

        if _dkey == "yolo11m":
            rankings_df.drop(columns=["detector_model"]).to_csv(_out_dir / "corrected_student_ahp_saw_rankings.csv", index=False)
            concordance_df.drop(columns=["detector_model"]).to_csv(_out_dir / "corrected_temporal_ahp_saw_concordance_holm.csv", index=False)
        rankings_df.to_csv(_out_dir / f"corrected_student_ahp_saw_rankings_{_dkey}.csv", index=False)
        concordance_df.to_csv(_out_dir / f"corrected_temporal_ahp_saw_concordance_holm_{_dkey}.csv", index=False)
        _all_temporal_concordance.append(concordance_df)

        print(f"\n=== WITHIN-SESSION PER-SEGMENT AHP-SAW RANKINGS ({_dkey}) ===")
        print(rankings_df.to_string(index=False))
        print(f"\n=== WITHIN-SESSION TEMPORAL CONCORDANCE WITH HOLM-BONFERRONI ({_dkey}) ===")
        print(concordance_df.to_string(index=False))

    if len(_all_temporal_concordance) > 1:
        pd.concat(_all_temporal_concordance, ignore_index=True).to_csv(
            MULTI_DETECTOR_PIPELINE_DIR / "multi_detector_temporal_concordance_holm_summary.csv", index=False
        )
else:
    logger.info(
        "[LOCKED] Within-Session Temporal Consistency and Robustness Analysis (ACTIVE_DETECTOR_MODEL = '%s') "
        "is locked (ENABLE_TEMPORAL_EXECUTION = False).",
        ACTIVE_DETECTOR_MODEL,
    )


=== DIRECT VS BIPROPORTIONAL IPF COMPARISON (yolo11m) ===
detector_model          Window_ID                        Window_Label Frame_Range  Window_Duration_Sec  Direct_Observed_Sec  Angkat_Tangan_Sec_Direct  Berdiri_Sec_Direct  Diam_Sec_Direct  Tidur_Sec_Direct  IPF_Observed_Sec  Angkat_Tangan_Sec_IPF  Berdiri_Sec_IPF  Diam_Sec_IPF  Tidur_Sec_IPF  Abs_Diff_Total_Sec  Rel_Diff_Total_Pct  Abs_Diff_Angkat_Tangan_Sec  Rel_Diff_Angkat_Tangan_Pct  Abs_Diff_Berdiri_Sec  Rel_Diff_Berdiri_Pct  Abs_Diff_Diam_Sec  Rel_Diff_Diam_Pct  Abs_Diff_Tidur_Sec  Rel_Diff_Tidur_Pct
       yolo11m           Segmen_1              Segmen 1 (00:00-05:00)      0-7499               300.00              6203.64                     41.80               19.56          4806.48           1335.80         6233.5065                23.9089          11.9845     5337.6425       859.9706            -29.8665              0.4814                     17.8911                     42.8017                7.5755               38.7296